# 01 · CsiNet 기준선 (실험 시작은 직접 클릭)

먼저 smoke check 셀만 실행하면 학습 20 batch와 validation 5 batch만 확인합니다. 이는 성능 결과가 아닙니다. 전체 학습은 아래 설정을 바꿔야만 시작됩니다. 공식 test split은 이 노트북에서 열지 않습니다.

In [ ]:
# Notebook는 각각 독립 실행 가능하도록 경로 설정을 포함합니다.
from pathlib import Path
import os, shutil, sys
import torch
from torch.utils.data import DataLoader
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT_OVERRIDE = None  # 필요하면 프로젝트 경로 지정
if PROJECT_ROOT_OVERRIDE: PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
DATA_DIR = PROJECT_ROOT / 'Data'
if IN_COLAB:
    ACTIVE_DATA_DIR = Path('/content/cost2100_indoor'); ACTIVE_DATA_DIR.mkdir(exist_ok=True)
    for name in ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']:
        src, dst = DATA_DIR / name, ACTIVE_DATA_DIR / name
        assert src.is_file(), f'필수 데이터 파일 없음: {src}'
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size: shutil.copy2(src, dst)
else:
    ACTIVE_DATA_DIR = DATA_DIR
sys.path.insert(0, str(PROJECT_ROOT))
from src.cost2100 import COST2100Dataset
from src.models.csinet import CsiNet
from src.metrics import sample_nmse_linear
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 200
train_ds = COST2100Dataset(ACTIVE_DATA_DIR / 'DATA_Htrainin.mat')
val_ds = COST2100Dataset(ACTIVE_DATA_DIR / 'DATA_Hvalin.mat')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
model = CsiNet(encoded_dim=512).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, eps=1e-7)
print('Project:', PROJECT_ROOT, '| device:', DEVICE)

In [ ]:
# 짧은 연결 확인. 20 train batch + 5 validation batch만 실행합니다.
model.train()
for step, x in enumerate(train_loader):
    x = x.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    pred = model(x)
    loss = (pred - x).square().mean()
    loss.backward()
    optimizer.step()
    if step == 19: break
model.eval()
with torch.no_grad():
    vals = []
    for step, x in enumerate(val_loader):
        x = x.to(DEVICE, non_blocking=True)
        vals.append(sample_nmse_linear(x, model(x)).mean().item())
        if step == 4: break
print('Smoke check complete; partial validation NMSE (not a result):', 10 * __import__('math').log10(sum(vals)/len(vals)))

## 전체 학습 설정

다음 셀의 `RUN_FULL_TRAINING`을 `True`로 바꾸고 실행할 때 1000 epoch 학습을 시작합니다. 로컬 RTX 3070에서도 시간이 오래 걸릴 수 있으므로 Colab GPU를 사용할 수 있습니다. 학습 완료 전 런타임 종료 시 체크포인트는 마지막 best 저장 시점까지만 남습니다. Smoke check 결과는 모델 품질 비교에 사용하지 않습니다.

In [ ]:
RUN_FULL_TRAINING = False  # 직접 True로 바꾸기 전에는 장시간 학습하지 않습니다.
EPOCHS = 100
RUN_DIR = PROJECT_ROOT / 'runs' / 'csinet_indoor_cr4'
RUN_DIR.mkdir(parents=True, exist_ok=True)
if RUN_FULL_TRAINING:
    # Smoke check updates are discarded; full training starts from fresh weights.
    model = CsiNet(encoded_dim=512).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, eps=1e-7)
    best_val = float('inf')
    for epoch in range(EPOCHS):
        model.train(); train_loss = 0.0
        for x in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = (pred - x).square().mean()
            loss.backward(); optimizer.step()
            train_loss += loss.item() * x.size(0)
        model.eval(); val_ratios = []
        with torch.no_grad():
            for x in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                val_ratios.append(sample_nmse_linear(x, model(x)).cpu())
        val_nmse = torch.cat(val_ratios).mean().item()
        train_loss /= len(train_ds)
        if val_nmse < best_val:
            best_val = val_nmse
            torch.save({'model': model.state_dict(), 'epoch': epoch + 1, 'val_nmse_linear': best_val}, RUN_DIR / 'best.pt')
        if (epoch + 1) % 10 == 0:
            print(f'epoch={epoch+1}/{EPOCHS} train_mse={train_loss:.6g} val_nmse_db={10*__import__("math").log10(val_nmse):.3f} best_db={10*__import__("math").log10(best_val):.3f}')
else:
    print('전체 학습을 시작하지 않았습니다. 설정을 검토한 뒤 직접 실행하세요.')